<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/03d-learning-rate-schedulers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import model_selection, preprocessing, pipeline
import torch
from torch import nn, optim, utils

# Preliminaries

## Define some utility functions

In [ ]:
def initialize_linear_layer(
    in_features,
    out_features,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    ):
    linear_layer = nn.Linear(in_features, out_features)

    if init_strategy_ is not None:
        if init_strategy_kwargs is None:
            init_strategy_kwargs = {}
        init_strategy_(linear_layer.weight, **init_strategy_kwargs)
        linear_layer.bias.data.fill_(0.0)

    return linear_layer


def make_mlp_classifier(
    input_size,
    hidden_sizes=None,
    output_size=2,
    activation_fn=None,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    batch_normalization=False
    ):
    modules = []
    hidden_sizes = [] if hidden_sizes is None else hidden_sizes
    for hidden_size in hidden_sizes:
        hidden_layer = initialize_linear_layer(
            input_size,
            hidden_size,
            init_strategy_,
            init_strategy_kwargs,
        )
        modules.append(hidden_layer)

        # batch normalization goes after the linear layer...
        if batch_normalization:
            modules.append(nn.BatchNorm1d(hidden_size))

        # ...but before the activation_fn!
        if activation_fn is not None:
            modules.append(activation_fn)
        input_size=hidden_size
    output_layer = initialize_linear_layer(
            input_size,
            output_size,
            init_strategy_,
            init_strategy_kwargs,
    )
    modules.append(output_layer)
    model_fn = nn.Sequential(*modules)
    return nn.CrossEntropyLoss(), model_fn


## Define the training loop

In [ ]:
def clip_gradients_(
    clip_grad_strategy,
    model_fn,
    clip_value=None,
    error_if_nonfinite=False,
    max_norm=None,
    norm_type=2.0
    ):
    if clip_grad_strategy == "value" and clip_value is not None:
        nn.utils.clip_grad_value_(
            model_fn.parameters(),
            clip_value
        )
    elif clip_grad_strategy == "norm" and max_norm is not None:
        nn.utils.clip_grad_norm_(
            model_fn.parameters(),
            max_norm,
            norm_type,
            error_if_nonfinite
        )
    else:
        raise NotImplementedError()


def compute_average_loss_per_batch(
    dataloader,
    criterion,
    model_fn
    ):
    total_loss = torch.zeros(1, 1)
    num_batches = len(dataloader)
    for features, targets in dataloader:
        predictions = model_fn(features)
        batch_loss = criterion(predictions, targets)
        total_loss += batch_loss
    average_loss_per_batch = total_loss / num_batches
    return average_loss_per_batch


def evaluate(model_fn, dataloader, metric, device="cpu"):
    model_fn.eval()
    metric.reset()
    with torch.inference_mode():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model_fn(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device="cpu",
    clip_grad_strategy=None,
    clip_value=None,
    error_if_nonfinite=False,
    lr_scheduler=None,
    log_epochs=1,
    max_epochs=1,
    max_norm=None,
    norm_type=2.0
    ):

    history = {
        "epoch": [],
        "average_train_loss": [],
        "average_val_loss": [],
        "lr": [],
    }
    num_train_batches = len(train_dataloader)
    for epoch in range(max_epochs):
        total_train_loss = torch.zeros(1, 1)
        model_fn = model_fn.train()
        for features, targets in train_dataloader:

            # forward pass
            features, targets = features.to(device), targets.to(device)
            predictions = model_fn(features)
            batch_loss = criterion(predictions, targets)
            total_train_loss += batch_loss

            # backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            if clip_grad_strategy is not None:
                clip_gradients_(
                    clip_grad_strategy,
                    model_fn,
                    clip_value,
                    error_if_nonfinite,
                    max_norm,
                    norm_type
                )
            optimizer.step()

        history["epoch"].append(epoch)

        average_train_loss_per_batch = total_train_loss / num_train_batches
        history["average_train_loss"].append(average_train_loss_per_batch.item())

        model_fn = model_fn.eval()
        with torch.inference_mode():
            average_val_loss_per_batch = compute_average_loss_per_batch(
                val_dataloader,
                criterion,
                model_fn
            )
        history["average_val_loss"].append(average_val_loss_per_batch.item())

        # update the learning rate after every training epoch
        if lr_scheduler is not None:
            history["lr"].append(lr_scheduler.get_last_lr()[-1])
            lr_scheduler.step()

        if (epoch + 1) % log_epochs == 0:
            print(
                f"Epoch {epoch},",
                f"Average train Loss {average_train_loss_per_batch.item():.4f},",
                f"Average val Loss {average_val_loss_per_batch.item():.4f}"
            )

    history_df = (
        pd.DataFrame.from_dict(history)
                    .set_index("epoch")
    )

    return history_df


## Load the data

In [ ]:
INPUT_SIZE = 784
OUTPUT_SIZE = 10
RANDOM_STATE = np.random.RandomState(42)


_train_data_df = pd.read_csv(
    "./sample_data/mnist_train_small.csv",
    header=None,
    names=["label"] + [f"p{i}" for i in range(INPUT_SIZE)],
)
train_data_df, val_data_df = model_selection.train_test_split(
    _train_data_df,
    random_state=RANDOM_STATE,
    stratify=_train_data_df.loc[:, "label"],
    test_size=0.1,
)

test_data_df = pd.read_csv(
    "./sample_data/mnist_test.csv",
    header=None,
    names=["label"] + [f"p{i}" for i in range(INPUT_SIZE)],
)

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


def series_to_tensor(s, dtype=torch.float32):
    arr = s.to_numpy()
    return array_to_tensor(arr, dtype)


features_preprocessor = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    preprocessing.FunctionTransformer(
        array_to_tensor,
        kw_args={
            "dtype":
            torch.float32
        }
    ),
)

target_preprocessor = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        series_to_tensor,
        kw_args={
            "dtype":
            torch.int64
        }
    ),
)

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2


# create the training dataset and dataloader
train_features_tensor = features_preprocessor.fit_transform(
    train_data_df.drop("label", axis=1)
)

train_target_tensor = target_preprocessor.fit_transform(
    train_data_df.loc[:, "label"]
)

train_dataset = utils.data.TensorDataset(
    train_features_tensor,
    train_target_tensor
)

mnist_train_dataloader = utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=True,
    num_workers=NUM_WORKERS,
    shuffle=True,
)

# create the validation dataset and dataloader
val_features_tensor = features_preprocessor.transform(
    val_data_df.drop("label", axis=1)
)

val_target_tensor = target_preprocessor.transform(
    val_data_df.loc[:, "label"]
)

val_dataset = utils.data.TensorDataset(
    val_features_tensor,
    val_target_tensor
)

mnist_val_dataloader = utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=True,
    num_workers=NUM_WORKERS,
    shuffle=False
)

# create the test dataset and dataloader
test_features_tensor = features_preprocessor.transform(
    test_data_df.drop("label", axis=1)
)

test_target_tensor = target_preprocessor.transform(
    test_data_df.loc[:, "label"]
)

test_dataset = utils.data.TensorDataset(
    test_features_tensor,
    test_target_tensor
)

mnist_test_dataloader = utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=False,
    num_workers=NUM_WORKERS,
    shuffle=False
)

# Learning Rate Schedulers

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HIDDEN_SIZE = int((2 / 3) * (INPUT_SIZE + OUTPUT_SIZE))
LEARNING_RATE = 1e-2
MAX_EPOCHS = 20

## Exponential Scheduling

In [ ]:
optim.lr_scheduler.ExponentialLR?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
model_fn = model_fn.to(DEVICE)


optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    dampening=0.0,
    weight_decay=0.0,
    nesterov=True,
)


lr_scheduler = optim.lr_scheduler.ExponentialLR(
    optimizer,
    gamma=0.9
)


exponential_lr_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    mnist_train_dataloader,
    mnist_val_dataloader,
    device=DEVICE,
    lr_scheduler=lr_scheduler,
    max_epochs=MAX_EPOCHS
)


In [ ]:
fig, axes = plt.subplots(1, 2)
_ = (
    exponential_lr_history_df.loc[:, ["average_train_loss", "average_val_loss"]]
            .plot(ax=axes[0], grid=True)
)
_ = (
    exponential_lr_history_df.loc[:, ["lr"]]
           .plot(ax=axes[1], grid=True)
)
fig.tight_layout()

## Cosine Annealing

In [ ]:
optim.lr_scheduler.CosineAnnealingLR?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
model_fn = model_fn.to(DEVICE)


optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    dampening=0.0,
    weight_decay=0.0,
    nesterov=True,
)


lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)


cosine_annealing_lr_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    mnist_train_dataloader,
    mnist_val_dataloader,
    device=DEVICE,
    lr_scheduler=lr_scheduler,
    max_epochs=MAX_EPOCHS
)


In [ ]:
fig, axes = plt.subplots(1, 2)
_ = (
    cosine_annealing_lr_history_df.loc[:, ["average_train_loss", "average_val_loss"]]
           .plot(ax=axes[0], grid=True)
)
_ = (
    cosine_annealing_lr_history_df.loc[:, ["lr"]]
           .plot(ax=axes[1], grid=True)
)
fig.tight_layout()

## Cosine Annealing with Warm Restarts

In [ ]:
optim.lr_scheduler.CosineAnnealingWarmRestarts?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
model_fn = model_fn.to(DEVICE)


optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    dampening=0.0,
    weight_decay=0.0,
    nesterov=True,
)


lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=3,
    T_mult=2,
    eta_min=1e-6,
)


cosine_annealing_warm_restarts_lr_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    mnist_train_dataloader,
    mnist_val_dataloader,
    device=DEVICE,
    lr_scheduler=lr_scheduler,
    max_epochs=MAX_EPOCHS
)


In [ ]:
fig, axes = plt.subplots(1, 2)
_ = (
    cosine_annealing_warm_restarts_lr_history_df.loc[:, ["average_train_loss", "average_val_loss"]]
           .plot(ax=axes[0], grid=True)
)
_ = (
    cosine_annealing_warm_restarts_lr_history_df.loc[:, ["lr"]]
           .plot(ax=axes[1], grid=True)
)
fig.tight_layout()

## 1-cycle Scheduling

In [ ]:
def fit_one_cycle_lr(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device="cpu",
    clip_grad_strategy=None,
    clip_value=None,
    error_if_nonfinite=False,
    one_cycle_lr_kwargs=None,
    log_epochs=1,
    max_epochs=1,
    max_norm=None,
    norm_type=2.0
    ):

    if one_cycle_lr_kwargs is None:
        one_cycle_lr_kwargs = {}
    lr_scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        epochs=max_epochs,
        steps_per_epoch=len(train_dataloader),
        **one_cycle_lr_kwargs
    )

    history = {
        "epoch": [],
        "average_train_loss": [],
        "average_val_loss": [],
        "lr": [],
    }
    num_train_batches = len(train_dataloader)
    for epoch in range(max_epochs):
        total_train_loss = torch.zeros(1, 1)
        model_fn = model_fn.train()
        for features, targets in train_dataloader:

            # forward pass
            features, targets = features.to(device), targets.to(device)
            predictions = model_fn(features)
            batch_loss = criterion(predictions, targets)
            total_train_loss += batch_loss

            # backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            if clip_grad_strategy is not None:
                clip_gradients_(
                    clip_grad_strategy,
                    model_fn,
                    clip_value,
                    error_if_nonfinite,
                    max_norm,
                    norm_type
                )
            optimizer.step()

            # update the learning rate here!
            lr_scheduler.step()

        history["epoch"].append(epoch)

        average_train_loss_per_batch = total_train_loss / num_train_batches
        history["average_train_loss"].append(average_train_loss_per_batch.item())

        model_fn = model_fn.eval()
        with torch.inference_mode():
            average_val_loss_per_batch = compute_average_loss_per_batch(
                val_dataloader,
                criterion,
                model_fn
            )
        history["average_val_loss"].append(average_val_loss_per_batch.item())

        # store learning rate afte every epoch!
        history["lr"].append(lr_scheduler.get_last_lr()[-1])

        if (epoch + 1) % log_epochs == 0:
            print(
                f"Epoch {epoch},",
                f"Average train Loss {average_train_loss_per_batch.item():.4f},",
                f"Average val Loss {average_val_loss_per_batch.item():.4f}"
            )

    history_df = (
        pd.DataFrame.from_dict(history)
                    .set_index("epoch")
    )

    return history_df

In [ ]:
optim.lr_scheduler.OneCycleLR?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
model_fn = model_fn.to(DEVICE)


optimizer = optim.SGD(
    model_fn.parameters(),
    lr=1e-3,
    momentum=0.9,
    dampening=0.0,
    weight_decay=0.0,
    nesterov=True,
)

lr_scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    epochs=MAX_EPOCHS,
    steps_per_epoch=len(mnist_train_dataloader),
    max_lr=LEARNING_RATE,
)

one_cycle_lr_history_df = fit_one_cycle_lr(
    criterion,
    model_fn,
    optimizer,
    mnist_train_dataloader,
    mnist_val_dataloader,
    device=DEVICE,
    one_cycle_lr_kwargs={
        "max_lr": LEARNING_RATE,
    },
    max_epochs=MAX_EPOCHS
)


In [ ]:
fig, axes = plt.subplots(1, 2)
_ = (
    one_cycle_lr_history_df.loc[:, ["average_train_loss", "average_val_loss"]]
           .plot(ax=axes[0], grid=True)
)
_ = (
    one_cycle_lr_history_df.loc[:, ["lr"]]
           .plot(ax=axes[1], grid=True)
)
fig.tight_layout()

## Performance Scheduling

In [ ]:
optim.lr_scheduler.ReduceLROnPlateau?

In [ ]:
def fit_reduce_lr_on_plateau(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device="cpu",
    clip_grad_strategy=None,
    clip_value=None,
    error_if_nonfinite=False,
    reduce_lr_on_plateau_kwargs=None,
    log_epochs=1,
    max_epochs=1,
    max_norm=None,
    norm_type=2.0
    ):

    history = {
        "epoch": [],
        "average_train_loss": [],
        "average_val_loss": [],
        "lr": [],
    }
    lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        **reduce_lr_on_plateau_kwargs
    )
    num_train_batches = len(train_dataloader)
    for epoch in range(max_epochs):
        total_train_loss = torch.zeros(1, 1)
        model_fn = model_fn.train()
        for features, targets in train_dataloader:

            # forward pass
            features, targets = features.to(device), targets.to(device)
            predictions = model_fn(features)
            batch_loss = criterion(predictions, targets)
            total_train_loss += batch_loss

            # backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            if clip_grad_strategy is not None:
                clip_gradients_(
                    clip_grad_strategy,
                    model_fn,
                    clip_value,
                    error_if_nonfinite,
                    max_norm,
                    norm_type
                )
            optimizer.step()

        history["epoch"].append(epoch)

        average_train_loss_per_batch = total_train_loss / num_train_batches
        history["average_train_loss"].append(average_train_loss_per_batch.item())

        model_fn = model_fn.eval()
        with torch.inference_mode():
            average_val_loss_per_batch = compute_average_loss_per_batch(
                val_dataloader,
                criterion,
                model_fn
            )
        history["average_val_loss"].append(average_val_loss_per_batch.item())

        # update the learning rate after every training epoch
        if lr_scheduler is not None:
            history["lr"].append(lr_scheduler.get_last_lr()[-1])
            lr_scheduler.step(average_val_loss_per_batch.item())

        if (epoch + 1) % log_epochs == 0:
            print(
                f"Epoch {epoch},",
                f"Average train Loss {average_train_loss_per_batch.item():.4f},",
                f"Average val Loss {average_val_loss_per_batch.item():.4f}"
            )

    history_df = (
        pd.DataFrame.from_dict(history)
                    .set_index("epoch")
    )

    return history_df


In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
model_fn = model_fn.to(DEVICE)


optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)


reduce_lr_on_plateau_kwargs = {
    "mode": "min",
    "patience": 2,
}

reduce_lr_on_plateau_history_df = fit_reduce_lr_on_plateau(
    criterion,
    model_fn,
    optimizer,
    mnist_train_dataloader,
    mnist_val_dataloader,
    device=DEVICE,
    reduce_lr_on_plateau_kwargs=reduce_lr_on_plateau_kwargs,
    max_epochs=MAX_EPOCHS,
)


In [ ]:
fig, axes = plt.subplots(1, 2)
_ = (
    reduce_lr_on_plateau_history_df.loc[:, ["average_train_loss", "average_val_loss"]]
           .plot(ax=axes[0], grid=True)
)
_ = (
    reduce_lr_on_plateau_history_df.loc[:, ["lr"]]
           .plot(ax=axes[1], grid=True)
)
fig.tight_layout()

## Exercise:

Create a MLP with three hidden layers, each with 100 neurons per layer, and train it for 10 epochs using the Adam optimizer with the following learning rate schedulers.

1. ExponentialLR
2. ReduceLROnPlateau
3. OneCycleLR

Plot the learning rate schedule, as well as the training and validation loss curves. Compare and contrast.